# Search

### The Knight's Tour Problem

The Knight’s Tour puzzle is the problem of finding a sequence of moves of a single knight on a chess board, such that the knight visits every square on the board exactly once. The knight may begin on any square and must move according to the normal rules for moving a knight in Chess.

You can find a very good description of the Knight's Tour problme in in its [Wikipedia entry](https://en.wikipedia.org/wiki/Knight%27s_tour)

A more mathematical presentation can  be  found  at
[mathworld.wolfram.com](https://mathworld.wolfram.com/KnightGraph.html)

For rules of movement in chess see http://en.wikipedia.org/wiki/Rules_of_chess#Basic_moves.

Normally, a standard 8x8 board is used, but essentially the same problem can be specified for any board dimensions.

A solution to the problem is a sequence of positions where each position can be reached by a knight's move from the previous position. To discover a solution we can generate different sequences until we find one that satisfies all the requirements of the problem. Because there are lots of different moves that we could try, the order that we consider the possibilities can make a big difference to how quickly a solution is found. 

Although we are using the Knight's Tour as an example, the same kind of thinking can be applied to a wide variety of puzzles and problems in computer science. We know the rules that a solution must follow, the conditions that must be satisfied, and there are many candidates to search through. We need a systematic and efficient way of approaching this search. 

The search is commonly thought of as a tree. Each node represents a _state_ that the puzzle can be in. 

#### Representing a state
In the Knight's Tour, a state is the current position of the Knight and a record of which squares have been visited so far. We can use a tuple to hold this information. In addition, we will record the number of moves made so far. So a state is the tuple
```
(move_number, (current_x_position, current_y_position),  full_board)
```
The board can be represented using a matrix, i.e. a list of lists.
If we have a board which measures 4 squares horizontally (x) and 3 squares vertically (y) this can be represented by
```
[[0, 0, 0, 0],
[0, 0, 0, 0],
[0, 0, 0, 0]]
```
A position on the board is then naturally represented by its x and y co-ordinate.
Notice that the usual way to write x and y co-ordinates is with the x value first, but that with list indexing we need to use the y value first.
For example, if the board is stored as a list of lists in a variable called `full_board` then position (1,2) would be accessed by `full_board[2][1]`.

When a position is visited we can change the 0 to a 1 to indicate this.
In order to show how the Knight has moved so far we will actually change the relevant value to show the number of the move which visited each square.
So if the Knight began at position (0,1) then moved to (2,0) and is now at (1,2) we would represent this with the board
```
[[0, 0, 2, 0],
[1, 0, 0, 0],
[0, 3, 0, 0]]
```
and the whole state would be given by 
```
(3, (1,2), [[0, 0, 2, 0], [1, 0, 0, 0], [0, 3, 0, 0]])
```

#### Possible moves
From the current position of (1,2) the Knight can either move to (0,0) or (3,1). It would be able to move to (2,0) except that this square has already been visited. Generally, a Knight can make up to eight different moves from its current position: either the x value is changed by 2 and the y value by 1, or the x value is changed by 1 and the y value by 2. 
The possible moves can be represented by the following eight tuples.
```
(1,2),  (2,1),  
(-1,2),  (2,-1), 
(1,-2), (-2,1), 
(-1,-2),  (-2,-1)
```
From the current position of (1,2) the moves (-1,-2) and (2,-1) take us to (0,0) and (3,1) respectively. The move (1,-2) would take us to (2,0), and all other moves result in co-ordinates outside of the board so are forbidden in this case. To check whether a move is viable we need to check whether either of the x or y co-ordinates that would result from applying it are negative or greater than the board size, and we need to check if the position has been previously visited. 

From (1,2) we attempt to apply the eight moves and discover that two of them are possible. Now we can decide which to try first.
Suppose we apply the move (-1,-2) first. Now the state is 
```
(4, (0,0), [[4, 0, 2, 0], [1, 0, 0, 0], [0, 3, 0, 0]])

```

#### Ordering the search
At this point we could decide to continue from the current state. For the next move we only have one option, which takes us to position (2,1), a couple of moves later there will be another choice to make. Eventually we will find that this attempt ends in failure. We can return to an earlier state and make different choices.
Alternatively, we could pause this attempt and generate the state which results from applying (2,-1) as the fourth move. Then we could generate the possible fifth moves from both states before moving on to the possible sixth moves and so on. 
These two approaches roughly describe a *depth first search* and a *breadth first search*.

The possible states are imagined as forming a tree. 
The root node represents the initial state, nothing has happened yet.
The children of this node represent the possible states after one move.
In general, the child of a node represents a state which can be reached from this node in one further move. 
Conceptually, we search through this tree of states until we find a "winning" state, one which satisfies our criteria for success. 
In the case of the Kinight's Tour puzzle, that means a state in which there are no remaining zeros in the lists representing the board.



## Brandon's Generic Search Algorithm

You should have downloaded a folder called "bbSearch" which contains a small Python package for carrying out a solution search for puzzles such as the Knight's Tour. 
The main function is `search()`.
We pass in arguments that define this puzzle as well as specifying the type of search we want to perform, and it will return a solution. 
You can open the files from bbSearch if you are interested, but here we will write the necessary functions to define the board and state and moves for the Knight's Tour, and to apply a move to a state as described above. 

In [13]:
from copy import deepcopy

# initialise a global variable for the size of the board
(BOARD_X, BOARD_Y) = (0,0)

# a global variable listing all Knight's moves
KNIGHT_MOVES = ((1,2), (2,1), (-1,2), (2,-1), (1,-2), (-2,1), (-1,-2), (-2,-1))

def knight_initial_state():
    """
        State is (move_number, (current_x_position, current_y_position),  full_board)
        Create an initial board filled with zeros
        The current move number is 0
        The initial x,y position is replaced with a zero because it's currently unknown
    """
    board = [[0 for x in range(BOARD_X)] for y in range(BOARD_Y)] 
    return (0,0,board)

def knight_initial_moves():
    """
        Generate possible initial positions i.e. all board positions.
    """
    moves = []
    for x in range(BOARD_X):
        for y in range(BOARD_Y):
            moves.append((x,y))
    return moves
    
def knight_following_moves(state):
    """
        Generate possible moves from a non-initial state
        For each of the eight knight moves
        - Check that the resulting position is within the board
        - Check that the position was not already visited
        If so, this move is possible
    """
    kx = state[1][0]
    ky = state[1][1]
    moves = []
    for move in KNIGHT_MOVES:
        newx = kx + move[0]    ## target x coord
        newy = ky + move[1]    ## target y coord

        ## If target square is on board and empty
        ## add it to the list of moves 
        if newx in range(BOARD_X) and newy in range(BOARD_Y):
            if state[2][newy][newx] == 0:
                moves = moves + [move]
    return moves

def knight_possible_moves(state):
    """
        Generate the next possible moves from this state.
        If currently in the start state, generates initial positions.
    """
    if state[0] == 0: 
        return knight_initial_moves()
    else:
        return knight_following_moves(state)

    
def knight_initial_successor(action):
    """
        Generate the initial state and apply the given action to it.
        
        Parameters:
            action: (x,y) co-ordinates of starting position
        
        Return:
            state: (move_number, (current_x_position, current_y_position),  full_board)
    """
    board = deepcopy(knight_initial_state()[2])
    board[action[1]][action[0]] = 1
    return (1, action, board)

def knight_successor_state(action, state):
    """
        Apply the action to the state.
        A state is (move_number, (current_x_position, current_y_position),  full_board)
        Return the new state following the action.
        The action (x,y) moves knight to (current_x_position + x, current_y_position + y)
        and we update the board representation accordingly.
        The move number is incremented.
        
        Parameters:
            action: (x,y) change to x and y values.
            state: the current state
        
        Return:
            state: the new state after applying the action
    """
    if state[0] == 0:
        newstate =  knight_initial_successor(action)
        return newstate
    board = deepcopy(state[2])
    xpos = state[1][0] + action[0]
    ypos = state[1][1] + action[1]
    movenum = state[0] + 1
    board[ypos][xpos] = movenum
    return (movenum, (xpos,ypos), board)       


def print_board_state(state):
    """
        Print the board from this state

        Parameters:
            state: (move_number, (current_x_position, current_y_position),  full_board)
    """
    full_board = state[2]
    for row in full_board:
       for square in row:
           print(" %2i" % square, end = '')
       print()
        
def knight_goal_state(state):
    """
        Check is this state a winning state?
    """
    winning_state = False
    
    if state[0] == BOARD_X * BOARD_Y:
        print( "\nGOAL STATE:" )
        print_board_state(state)
        winning_state = True
        return winning_state
    else:
        return winning_state


def knight_initialise(arg):
    """
        Set the global X and Y sizes of the board.
        
        Parameters:
            arg: (X, Y) tuple specifying board size
    """
    global BOARD_X, BOARD_Y
    BOARD_X, BOARD_Y = arg
    
def knight_print_problem_info():
    """
        Print basic info about this puzzle
    """
    print("The Knight's Tour (", BOARD_X, "x", BOARD_Y, "board)")


We now package these functions up ready to be passed into the search. 

In [14]:
problem = ( knight_initialise,
         knight_print_problem_info,
         knight_initial_state(),
         knight_possible_moves,
         knight_successor_state,
         knight_goal_state
      )

We can now call the search function itself.
This takes 5 paremeters:
* The problem specification (as given in the last cell)
* A configuration parameter (in this case the board dimensions)
* The choice of search type
* A _node_limit_, which is the maximum number of nodes that will
  be created in the search tree before the search is aborted.
  (Note that each node corresponds to a state, but states can potentially
  occur at many nodes as there could be many ways to get to the same state.)
* A search option list. Look at the header of bbSearch/queue_search.py
  if you are interested to see what these options are.

In [24]:
import gc
import bbSearch
gc.collect()

my_search = bbSearch.search(problem, (4,6), 'depth_first',  1000000, [])

======= Running Brandon's Python Queue-Based Search Procedure =======
The Knight's Tour ( 4 x 6 board)
Strategy: depth_first
Search Limit: max_nodes = 1000000
Options: []
*** starting search ***
0  1000  2000  3000  4000  5000  6000  7000  8000  9000  10000  11000  12000  13000  14000  15000  16000  17000  18000  19000  20000  21000  22000  23000  24000  25000  26000  27000  28000  29000  30000  31000  32000  33000  34000  35000  36000  37000  38000  39000  40000  41000  42000  43000  44000  45000  46000  47000  48000  49000  50000  51000  52000  53000  54000  55000  56000  57000  58000  59000  60000  61000  62000  63000  64000  65000  66000  67000  68000  69000  70000  71000  72000  73000  74000  75000  76000  77000  78000  79000  80000  81000  82000  83000  84000  85000  86000  87000  88000  89000  90000  91000  92000  93000  
GOAL STATE:
  1 10 21 18
 20 17 12  9
 11  2 19 22
 16 13  8  5
  3  6 23 14
 24 15  4  7

:-)) *SUCCESS* ((-:
The action path to the solution is:
(0, 0), (1, 

In [33]:
isinstance(my_search, list)

True

## Excercise 1
Test and  compare  the  performance  of breadth first search (using argument 'breadth_first' for the 3rd parameter) and depth first search (use argument 'depth_first') on the Knight’s Tour puzzle for each of the board sizes 3x4, 4x4, 4x5 and 5x5.
Construct a table, which clearly shows all significant information generated by each of the search tests (i.e. result of the search, nodes generated, time taken, length of path).

Which of the strategies breadth first search and depth first search is best for solving the Knight’s Tour problem.
Can you explain this? 


In [54]:
import pandas as pd
import time

board_sizes = [(3,4), (4,4), (4,5), (5,5)]
search_strategy = ['breadth_first', 'depth_first']
results = {}
i = 0

for board in board_sizes:
    for strategy in search_strategy:
        time_start = time.perf_counter()
        attempt = bbSearch.search(problem, board, strategy,  100000, [])
        time_taken = time.perf_counter() - time_start
        
        if isinstance(attempt,list):
            action_path = attempt
        else:
            action_path = []
        
        print('writing results')
        results[i] = {'board' : board, 
                        'strategy' : strategy,
                        'success' : isinstance(attempt,list),
                        'path_length' : len(action_path),
                        'time' : time_taken}
        i+=1

df = pd.DataFrame(results).T

======= Running Brandon's Python Queue-Based Search Procedure =======
The Knight's Tour ( 3 x 4 board)
Strategy: breadth_first
Search Limit: max_nodes = 100000
Options: []
*** starting search ***
0  
GOAL STATE:
  1 12  3
  4  9  6
  7  2 11
 10  5  8

:-)) *SUCCESS* ((-:
The action path to the solution is:
(0, 0), (1, 2), (1, -2), (-2, 1), (1, 2), (1, -2), (-2, 1), (2, 1), (-1, -2), (-1, 2), (2, -1), (-1, -2)
Path length = 12
Total nodes tested = 746
Time taken = 0.01918160000059288 seconds

writing results
======= Running Brandon's Python Queue-Based Search Procedure =======
The Knight's Tour ( 3 x 4 board)
Strategy: depth_first
Search Limit: max_nodes = 100000
Options: []
*** starting search ***
0  
GOAL STATE:
  1 12  3
  4  9  6
  7  2 11
 10  5  8

:-)) *SUCCESS* ((-:
The action path to the solution is:
(0, 0), (1, 2), (1, -2), (-2, 1), (1, 2), (1, -2), (-2, 1), (2, 1), (-1, -2), (-1, 2), (2, -1), (-1, -2)
Path length = 12
Total nodes tested = 13
Time taken = 0.002225500000349711

board                   success               path_length  \
strategy breadth_first depth_first breadth_first depth_first breadth_first   
0               (3, 4)         NaN          True         NaN            12   
1                  NaN      (3, 4)           NaN        True           NaN   
2               (4, 4)         NaN         False         NaN             0   
3                  NaN      (4, 4)           NaN       False           NaN   
4               (4, 5)         NaN         False         NaN             0   
5                  NaN      (4, 5)           NaN        True           NaN   
6               (5, 5)         NaN         False         NaN             0   
7                  NaN      (5, 5)           NaN        True           NaN   

                              time              
strategy depth_first breadth_first depth_first  
0                NaN      0.019606         NaN  
1                 12           NaN    0.002271  
2                NaN      1.203061         NaN  
3                  0           NaN    5.364338  
4                NaN     77.048546         NaN  
5                 20           NaN    0.149147  
6                NaN    195.897222         NaN  
7                 25           NaN    4.652056

In [58]:
df

,board,strategy,success,path_length,time
0,"(3, 4)",breadth_first,True,12,0.019606
1,"(3, 4)",depth_first,True,12,0.002271
2,"(4, 4)",breadth_first,False,0,1.203061
3,"(4, 4)",depth_first,False,0,5.364338
4,"(4, 5)",breadth_first,False,0,77.048546
5,"(4, 5)",depth_first,True,20,0.149147
6,"(5, 5)",breadth_first,False,0,195.897222
7,"(5, 5)",depth_first,True,25,4.652056


## Exercise 2

A nice interactive tool for exploring the behaviour of various search algorithms can be found online at:

http://qiao.github.io/PathFinding.js/visual/

You can use this to investigate how the different algorithms behave in different situations.
* Create obstacles which block the straight path from Start to Goal.
* Try at least the Breadth First, Best First, Dijkstra (this is also called Uniform Cost) and A* algorithms.
* Try creating obstacle structures such that the goal cannot be reached by progressing uniformly towards it, but requires movement away from the goal for some stages of the path.